# Neo4j Graph Analytics on Snowflake

**Prerequisite:** Neo4j Graph Analytics installed from Snowflake Marketplace  
**Data:** Process mining node/relationship tables from Notebook 02  
**Features:** Community detection (WCC, Louvain), PageRank, similarity - all via SQL

In [ ]:
USE DATABASE CDSB_DEMO;
USE SCHEMA RAW;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
!pip install plotly --quiet

In [ ]:
import pandas as pd
import plotly.express as px
import streamlit.components.v1 as components
from neo4j_viz.snowflake import from_snowflake
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print('Connected to', session.get_current_database(), session.get_current_schema())

## 1. Setup - Grant Neo4j Access to Our Data
Neo4j runs as a Native App inside Snowflake - your data never leaves

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE DATABASE ROLE IF NOT EXISTS CDSB_DEMO.NEO4J_ROLE;
GRANT USAGE ON DATABASE CDSB_DEMO TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT USAGE ON SCHEMA CDSB_DEMO.RAW TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT SELECT ON ALL TABLES IN SCHEMA CDSB_DEMO.RAW TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT SELECT ON FUTURE TABLES IN SCHEMA CDSB_DEMO.RAW TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT SELECT ON ALL VIEWS IN SCHEMA CDSB_DEMO.RAW TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT SELECT ON FUTURE VIEWS IN SCHEMA CDSB_DEMO.RAW TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT CREATE TABLE ON SCHEMA CDSB_DEMO.RAW TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE;
GRANT DATABASE ROLE CDSB_DEMO.NEO4J_ROLE TO APPLICATION SE_SNOW_NEO4J_GRAPH_ANALYTICS

In [ ]:
for tbl in ['CASE_NODES','HANDLER_NODES','REGION_NODES','HANDLED_BY_RELS','IN_REGION_RELS']:
    n = session.sql(f'SELECT COUNT(*) as N FROM {tbl}').to_pandas().iloc[0,0]
    cols = session.sql(f"SELECT COLUMN_NAME, DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME='{tbl}' AND TABLE_SCHEMA='RAW'").to_pandas()
    types = ', '.join(f'{r.COLUMN_NAME}({r.DATA_TYPE})' for _, r in cols.iterrows())
    print(f'{tbl}: {n:,} rows  [{types}]')
print('\nAll columns are numeric - ready for Neo4j!')

In [ ]:
session.sql('CREATE OR REPLACE VIEW VIZ_CASE_SAMPLE AS SELECT NODEID FROM CASE_NODES SAMPLE (200 ROWS)').collect()
session.sql('CREATE OR REPLACE VIEW VIZ_HANDLED_BY AS SELECT r.SOURCENODEID, r.TARGETNODEID, r.WEIGHT FROM HANDLED_BY_RELS r JOIN VIZ_CASE_SAMPLE v ON r.SOURCENODEID = v.NODEID').collect()
session.sql('CREATE OR REPLACE VIEW VIZ_IN_REGION AS SELECT r.SOURCENODEID, r.TARGETNODEID, r.WEIGHT FROM IN_REGION_RELS r JOIN VIZ_CASE_SAMPLE v ON r.SOURCENODEID = v.NODEID').collect()
session.sql('GRANT SELECT ON ALL VIEWS IN SCHEMA CDSB_DEMO.RAW TO DATABASE ROLE CDSB_DEMO.NEO4J_ROLE').collect()
print('Created VIZ_CASE_SAMPLE (200 cases), VIZ_HANDLED_BY, VIZ_IN_REGION')

### Interactive Knowledge Graph
Visualise 200 sampled cases with their handler and region connections using `neo4j-viz`

In [ ]:
viz_graph = from_snowflake(session, {
    'defaultTablePrefix': 'CDSB_DEMO.RAW',
    'nodeTables': ['VIZ_CASE_SAMPLE', 'HANDLER_NODES', 'REGION_NODES'],
    'relationshipTables': {
        'VIZ_HANDLED_BY': {
            'sourceTable': 'VIZ_CASE_SAMPLE',
            'targetTable': 'HANDLER_NODES'
        },
        'VIZ_IN_REGION': {
            'sourceTable': 'VIZ_CASE_SAMPLE',
            'targetTable': 'REGION_NODES'
        }
    }
})

for node in viz_graph.nodes:
    node.caption = str(node.id)

html_obj = viz_graph.render()
components.html(html_obj.data, height=600)

## 2. Community Detection - Weakly Connected Components
Find disconnected groups of cases and handlers

In [ ]:
CALL SE_SNOW_NEO4J_GRAPH_ANALYTICS.GRAPH.WCC('CPU_X64_XS', {
    'defaultTablePrefix': 'CDSB_DEMO.RAW',
    'project': {
        'nodeTables': ['CASE_NODES', 'HANDLER_NODES', 'REGION_NODES'],
        'relationshipTables': {
            'HANDLED_BY_RELS': {
                'sourceTable': 'CASE_NODES',
                'targetTable': 'HANDLER_NODES',
                'orientation': 'UNDIRECTED'
            },
            'IN_REGION_RELS': {
                'sourceTable': 'CASE_NODES',
                'targetTable': 'REGION_NODES',
                'orientation': 'UNDIRECTED'
            }
        }
    },
    'compute': { 'consecutiveIds': true },
    'write': [
        {'nodeLabel': 'CASE_NODES', 'outputTable': 'CASE_WCC_RESULTS'},
        {'nodeLabel': 'HANDLER_NODES', 'outputTable': 'HANDLER_WCC_RESULTS'},
        {'nodeLabel': 'REGION_NODES', 'outputTable': 'REGION_WCC_RESULTS'}
    ]
})

In [ ]:
GRANT SELECT ON TABLE CASE_WCC_RESULTS TO ROLE ACCOUNTADMIN;
GRANT SELECT ON TABLE HANDLER_WCC_RESULTS TO ROLE ACCOUNTADMIN;
GRANT SELECT ON TABLE REGION_WCC_RESULTS TO ROLE ACCOUNTADMIN;

In [ ]:
df_wcc = session.sql("""
    SELECT 'Case' as NODE_TYPE, COMPONENT, COUNT(*) as NODES FROM CASE_WCC_RESULTS GROUP BY COMPONENT
    UNION ALL
    SELECT 'Handler', COMPONENT, COUNT(*) FROM HANDLER_WCC_RESULTS GROUP BY COMPONENT
    UNION ALL
    SELECT 'Region', COMPONENT, COUNT(*) FROM REGION_WCC_RESULTS GROUP BY COMPONENT
    ORDER BY NODES DESC LIMIT 15
""").to_pandas()
print('Connected Components by Node Type:')
print(df_wcc.to_string(index=False))

## 3. PageRank - Find the Most Central Handlers
Who are the most connected / influential actors in the service delivery network?

In [ ]:
CALL SE_SNOW_NEO4J_GRAPH_ANALYTICS.GRAPH.PAGE_RANK('CPU_X64_XS', {
    'defaultTablePrefix': 'CDSB_DEMO.RAW',
    'project': {
        'nodeTables': ['CASE_NODES', 'HANDLER_NODES', 'REGION_NODES'],
        'relationshipTables': {
            'HANDLED_BY_RELS': {
                'sourceTable': 'CASE_NODES',
                'targetTable': 'HANDLER_NODES'
            },
            'IN_REGION_RELS': {
                'sourceTable': 'CASE_NODES',
                'targetTable': 'REGION_NODES'
            }
        }
    },
    'compute': { 'maxIterations': 20, 'dampingFactor': 0.85 },
    'write': [
        {'nodeLabel': 'HANDLER_NODES', 'outputTable': 'HANDLER_PAGERANK_RESULTS'},
        {'nodeLabel': 'REGION_NODES', 'outputTable': 'REGION_PAGERANK_RESULTS'}
    ]
})

In [ ]:
GRANT SELECT ON TABLE HANDLER_PAGERANK_RESULTS TO ROLE ACCOUNTADMIN;
GRANT SELECT ON TABLE REGION_PAGERANK_RESULTS TO ROLE ACCOUNTADMIN;

In [ ]:
session.sql('CREATE OR REPLACE VIEW VIZ_HANDLER_PR AS SELECT NODEID, PAGERANK FROM HANDLER_PAGERANK_RESULTS').collect()
session.sql('CREATE OR REPLACE VIEW VIZ_REGION_PR AS SELECT NODEID, PAGERANK FROM REGION_PAGERANK_RESULTS').collect()
session.sql('CREATE OR REPLACE VIEW VIZ_HANDLED_BY_SAMPLE AS SELECT SOURCENODEID, TARGETNODEID, WEIGHT FROM HANDLED_BY_RELS WHERE SOURCENODEID IN (SELECT NODEID FROM VIZ_CASE_SAMPLE)').collect()
session.sql('CREATE OR REPLACE VIEW VIZ_IN_REGION_SAMPLE AS SELECT SOURCENODEID, TARGETNODEID, WEIGHT FROM IN_REGION_RELS WHERE SOURCENODEID IN (SELECT NODEID FROM VIZ_CASE_SAMPLE)').collect()

viz_pr = from_snowflake(session, {
    'defaultTablePrefix': 'CDSB_DEMO.RAW',
    'nodeTables': ['VIZ_CASE_SAMPLE', 'VIZ_HANDLER_PR', 'VIZ_REGION_PR'],
    'relationshipTables': {
        'VIZ_HANDLED_BY_SAMPLE': {
            'sourceTable': 'VIZ_CASE_SAMPLE',
            'targetTable': 'VIZ_HANDLER_PR'
        },
        'VIZ_IN_REGION_SAMPLE': {
            'sourceTable': 'VIZ_CASE_SAMPLE',
            'targetTable': 'VIZ_REGION_PR'
        }
    }
})

sizes = {}
for node in viz_pr.nodes:
    pr_val = node.properties.get('PAGERANK', 0)
    sizes[node.id] = pr_val if pr_val else 1
    node.caption = str(node.id)

viz_pr.resize_nodes(sizes=sizes, node_radius_min_max=(5, 80))

html_pr = viz_pr.render()
components.html(html_pr.data, height=600)

### PageRank Bar Chart

In [ ]:
df_pr = session.sql("""
    SELECT 'Handler' as NODE_TYPE, pr.NODEID, ROUND(pr.PAGERANK, 6) as PAGERANK, n.NODE_ID
    FROM HANDLER_PAGERANK_RESULTS pr
    JOIN PROCESS_NODES n ON pr.NODEID = n.NODEID
    UNION ALL
    SELECT 'Region', pr.NODEID, ROUND(pr.PAGERANK, 6), n.NODE_ID
    FROM REGION_PAGERANK_RESULTS pr
    JOIN PROCESS_NODES n ON pr.NODEID = n.NODEID
    ORDER BY PAGERANK DESC LIMIT 15
""").to_pandas()
print('Most Central Actors (PageRank):')
print(df_pr.to_string(index=False))

fig = px.bar(df_pr, x='NODE_ID', y='PAGERANK', color='NODE_TYPE',
             title='PageRank: Most Central Actors in Service Delivery Network')
fig.update_layout(template='plotly_white')
fig.show()

## 4. Louvain Community Detection
Find tightly-connected communities of handlers and cases

In [ ]:
CALL SE_SNOW_NEO4J_GRAPH_ANALYTICS.GRAPH.LOUVAIN('CPU_X64_XS', {
    'defaultTablePrefix': 'CDSB_DEMO.RAW',
    'project': {
        'nodeTables': ['CASE_NODES', 'HANDLER_NODES', 'REGION_NODES'],
        'relationshipTables': {
            'HANDLED_BY_RELS': {
                'sourceTable': 'CASE_NODES',
                'targetTable': 'HANDLER_NODES',
                'orientation': 'UNDIRECTED'
            },
            'IN_REGION_RELS': {
                'sourceTable': 'CASE_NODES',
                'targetTable': 'REGION_NODES',
                'orientation': 'UNDIRECTED'
            }
        }
    },
    'compute': {},
    'write': [
        {'nodeLabel': 'CASE_NODES', 'outputTable': 'CASE_LOUVAIN_RESULTS'},
        {'nodeLabel': 'HANDLER_NODES', 'outputTable': 'HANDLER_LOUVAIN_RESULTS'},
        {'nodeLabel': 'REGION_NODES', 'outputTable': 'REGION_LOUVAIN_RESULTS'}
    ]
})

In [ ]:
GRANT SELECT ON TABLE CASE_LOUVAIN_RESULTS TO ROLE ACCOUNTADMIN;
GRANT SELECT ON TABLE HANDLER_LOUVAIN_RESULTS TO ROLE ACCOUNTADMIN;
GRANT SELECT ON TABLE REGION_LOUVAIN_RESULTS TO ROLE ACCOUNTADMIN;

In [ ]:
session.sql('CREATE OR REPLACE VIEW VIZ_CASE_LOUVAIN AS SELECT NODEID, COMMUNITY FROM CASE_LOUVAIN_RESULTS WHERE NODEID IN (SELECT NODEID FROM VIZ_CASE_SAMPLE)').collect()
session.sql('CREATE OR REPLACE VIEW VIZ_HANDLER_LOUVAIN AS SELECT NODEID, COMMUNITY FROM HANDLER_LOUVAIN_RESULTS').collect()
session.sql('CREATE OR REPLACE VIEW VIZ_REGION_LOUVAIN AS SELECT NODEID, COMMUNITY FROM REGION_LOUVAIN_RESULTS').collect()

viz_louv = from_snowflake(session, {
    'defaultTablePrefix': 'CDSB_DEMO.RAW',
    'nodeTables': ['VIZ_CASE_LOUVAIN', 'VIZ_HANDLER_LOUVAIN', 'VIZ_REGION_LOUVAIN'],
    'relationshipTables': {
        'VIZ_HANDLED_BY_SAMPLE': {
            'sourceTable': 'VIZ_CASE_LOUVAIN',
            'targetTable': 'VIZ_HANDLER_LOUVAIN'
        },
        'VIZ_IN_REGION_SAMPLE': {
            'sourceTable': 'VIZ_CASE_LOUVAIN',
            'targetTable': 'VIZ_REGION_LOUVAIN'
        }
    }
})

viz_louv.color_nodes(property='COMMUNITY', override=True)
for node in viz_louv.nodes:
    node.caption = str(node.id)

html_louv = viz_louv.render()
components.html(html_louv.data, height=600)

### Louvain Communities Bar Chart

In [ ]:
df_communities = session.sql("""
    SELECT 'Case' as NODE_TYPE, l.COMMUNITY, COUNT(*) as MEMBERS FROM CASE_LOUVAIN_RESULTS l GROUP BY l.COMMUNITY
    UNION ALL
    SELECT 'Handler', l.COMMUNITY, COUNT(*) FROM HANDLER_LOUVAIN_RESULTS l GROUP BY l.COMMUNITY
    UNION ALL
    SELECT 'Region', l.COMMUNITY, COUNT(*) FROM REGION_LOUVAIN_RESULTS l GROUP BY l.COMMUNITY
    ORDER BY MEMBERS DESC LIMIT 20
""").to_pandas()
print('Communities Detected:')
print(df_communities.to_string(index=False))

df_communities['COMMUNITY'] = df_communities['COMMUNITY'].astype(str)
fig = px.bar(df_communities, x='COMMUNITY', y='MEMBERS', color='NODE_TYPE',
             title='Louvain Communities: Handler-Case Clusters')
fig.update_layout(template='plotly_white', xaxis_title='Community ID')
fig.show()

## 5. Combine Graph Features with Process Mining Results
Join community/centrality data back to original cases for deeper analysis

In [ ]:
df_enriched = session.sql("""
    WITH case_summary AS (
        SELECT CASE_ID,
               LISTAGG(ACTIVITY, ' > ') WITHIN GROUP (ORDER BY EVENT_TIME) as PATH,
               DATEDIFF('hour', MIN(EVENT_TIME), MAX(EVENT_TIME)) as DURATION_HOURS,
               MAX(CHANNEL) as CHANNEL, MAX(REQUEST_TYPE) as REQUEST_TYPE, MAX(ACTOR) as HANDLER
        FROM SERVICE_REQUEST_EVENTS
        WHERE ACTOR NOT IN ('System', 'Triage_Team', 'Escalation_Manager')
        GROUP BY CASE_ID
    ),
    handler_rank AS (
        SELECT n.NODE_ID as HANDLER, ROUND(pr.PAGERANK, 6) as PAGERANK
        FROM HANDLER_PAGERANK_RESULTS pr
        JOIN PROCESS_NODES n ON pr.NODEID = n.NODEID
        WHERE n.NODE_TYPE = 'Handler'
    )
    SELECT cs.*, hr.PAGERANK as HANDLER_CENTRALITY
    FROM case_summary cs LEFT JOIN handler_rank hr ON cs.HANDLER = hr.HANDLER
    ORDER BY DURATION_HOURS DESC LIMIT 20
""").to_pandas()

print('Enriched Case Data (graph features + process mining):')
df_enriched

## Key Takeaways

| Algorithm | Use Case | What It Found |
|-----------|----------|--------------|
| WCC | Find disconnected case groups | Isolated case clusters that may indicate process silos |
| PageRank | Identify key handlers | Most central actors in service delivery |
| Louvain | Community detection | Handler-case communities revealing team dynamics |

**Neo4j Graph Analytics runs inside Snowflake as a Native App**  
- No data movement - your data stays in Snowflake  
- SQL interface - no Cypher needed  
- Results write back to Snowflake tables for joining, dashboarding, ML  
- Perfect for fraud detection, entity resolution, network analysis